# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library. We'll use the Croissant schema to access data and metadata in a reproducible fashion—always referencing data elements by their `@id` fields.

### Dataset Source
The dataset source is provided via the Croissant schema URL below.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, fields, and column `@id`s.

Below we print all available record sets and their fields. This allows you to select appropriate `@id`s for subsequent extraction.

In [ ]:
# List all record sets and their fields (referenced by @id)
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (field @id: {field.id}, dataType: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis, using the record set and field `@id`s identified above.

For demonstration, we'll load **all record sets** and create a DataFrame for each (keyed by their record set `@id`).

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for rs_id in record_set_ids:
    # list(dataset.records(record_set=...)) yields dicts with field @ids as keys
    recs = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(recs)
    print(f"DataFrame for record set {rs_id} has {len(dataframes[rs_id])} rows and {len(dataframes[rs_id].columns)} columns.")

# Pick the first record set for further analysis, or replace with a specific @id of your choice from above
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"\nColumns in record set ({example_record_set_id}):")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate basic EDA steps: filtering numeric data, normalizing, and grouping. 

Please update the variable assignments below to match particular fields (`@id`s) you want to analyze—using the list from "Data Extraction" above.

In [ ]:
# Example: Select a numeric field in the chosen record set for analysis
record_set_id = example_record_set_id
df = dataframes[record_set_id]

# Choose a numeric field by its @id, e.g., 'log_likelihood' if present
# Use the previous cell's output to select an appropriate @id
numeric_fields = [col for col in df.columns if df[col].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields: {numeric_fields}")

if numeric_fields:
    numeric_field = numeric_fields[0]   # Change to another field @id if relevant
    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0

    # Filter records where the numeric field exceeds its mean
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize this field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if present
    # Select a group-by field (non-numeric), e.g., 'region' or similar
    group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if group_fields:
        group_field = group_fields[0]  # Change as appropriate
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No numeric fields found in the selected record set.")

## 5. Visualization
Below we visualize the distribution of numeric data, and relationships with potential categorical fields, using `matplotlib` and `seaborn`.

Update the `numeric_field` and optionally `group_field` below with fields of interest from your previous cells.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping field identified, display boxplot
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field with sufficient data to visualize.")

## 6. Conclusion
We have demonstrated how to:
- Load a FAIR-compliant dataset via its Croissant schema and examine record sets using `mlcroissant`
- Reference all data elements by their unique `@id`
- Extract tabular data into pandas DataFrames, filter, normalize, and group records
- Visualize data distributions and grouped statistics

**Next steps:**
- Further explore variable relationships, missing data patterns, and model results
- Use the rich metadata provided for ethical and methodological documentation

Remember to consult field definitions and `@id` mappings in the data overview section to ensure correct references throughout your analysis.